In [38]:
import json
import pandas as pd

# I upload the trial proceedings in jsonl format to keep the raw downloaded data as it is. Beware that this is jsonl and not json so I need to use json.loads(line) to upload
proceedings_raw = [json.loads(line) for line in open("trial_proceedings.jsonl", "r", encoding="utf-8")]

# Columns are nested in JSONL. I expand or 'flatten' the columns from their nested format.
proceedings_raw2 = pd.json_normalize(proceedings_raw, sep=".")
proceedings_raw2.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19100 entries, 0 to 19099
Data columns (total 51 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   trial_number                                            19100 non-null  object 
 1   last_modified_date_time                                 19100 non-null  object 
 2   respondent_data                                         0 non-null      float64
 3   derivation_petitioner_data                              0 non-null      float64
 4   raw_data                                                0 non-null      object 
 5   trial_meta_data.petition_filing_date                    19098 non-null  object 
 6   trial_meta_data.accorded_filing_date                    19076 non-null  object 
 7   trial_meta_data.trial_last_modified_date_time           16619 non-null  object 
 8   trial_meta_data.trial_last_modified_

In [39]:
# I check if any trial challenges multiple patents
PROCEEDING_COL = "trial_number"
PATENT_COL = "patent_owner_data.patent_number"

patents_per_proc = (
    proceedings_raw2.dropna(subset=[PROCEEDING_COL, PATENT_COL])
      .groupby(PROCEEDING_COL)[PATENT_COL]
      .nunique()
)

# proceedings with multiple patents
multi_pat_procs = patents_per_proc[patents_per_proc > 1]

print("Proceedings with >1 patent:", multi_pat_procs.shape[0])
print("Share of all proceedings:", (multi_pat_procs.shape[0] / patents_per_proc.shape[0]))
print(multi_pat_procs.head(10))

Proceedings with >1 patent: 0
Share of all proceedings: 0.0
Series([], Name: patent_owner_data.patent_number, dtype: int64)


In [40]:
# I want to select the columns I am interested in. The problems is when I expand the netted data set 51 columns appear even tough USPTO website says there are only 26 columns.
# My guess is there are actually more variables of the trial proceedings that USPTO hasn't uploaded yet or decided not to upload since columns look empty.
# I used a little help from ChatGpt to make an educated guess to determine which columns are the columns I actually see at USPTO website.

cols = [
    "trial_number",
    "trial_meta_data.trial_status_category",
    "trial_meta_data.petition_filing_date",
    "trial_meta_data.institution_decision_date",
    "trial_meta_data.latest_decision_date",
    "trial_meta_data.termination_date",
    "patent_owner_data.application_number_text",
    "patent_owner_data.patent_number",
    "patent_owner_data.grant_date",
    "patent_owner_data.technology_center_number",
    "patent_owner_data.inventor_name",
    "patent_owner_data.group_art_unit_number",
    "patent_owner_data.patent_owner_name",
    "patent_owner_data.real_party_in_interest_name",
    "regular_petitioner_data.real_party_in_interest_name",
]

# This is now more like the table I see at the USPTO website
proceedings_raw3 = proceedings_raw2[cols].copy()

In [41]:
# Now I want to understand these variables so I am checking the unique entries, frequency tables of each column (just change the column name in the code)
proceedings_raw3[cols].nunique(dropna=True)
proceedings_raw3["trial_meta_data.institution_decision_date"].value_counts(dropna=False)

# I drop the columns that I don't like such as 'trial_meta_data.latest_decision_date'
proceedings_raw3 = proceedings_raw3.drop(columns=["trial_meta_data.latest_decision_date"])
proceedings_raw3.head(-10)

,trial_number,trial_meta_data.trial_status_category,trial_meta_data.petition_filing_date,trial_meta_data.institution_decision_date,trial_meta_data.termination_date,patent_owner_data.application_number_text,patent_owner_data.patent_number,patent_owner_data.grant_date,patent_owner_data.technology_center_number,patent_owner_data.inventor_name,patent_owner_data.group_art_unit_number,patent_owner_data.patent_owner_name,patent_owner_data.real_party_in_interest_name,regular_petitioner_data.real_party_in_interest_name
0,IPR2026-00191,Pending,2026-01-02,None,None,16832820,11031677,2021-06-08,2600,Carles PUENTE BALIARDA et al,2643,None,None,Resmed Corp.
1,IPR2026-00156,Pending,2025-12-31,None,None,15403146,9772193,2017-09-26,2600,Ehud MENDELSON,2648,None,None,Google LLC
2,IPR2026-00184,Pending,2025-12-30,None,None,18885510,12231426,2025-02-18,2400,Jason Crabtree et al,2495,None,None,Microsoft Corporation
3,IPR2026-00182,Pending,2025-12-30,None,None,18885474,12218934,2025-02-04,2400,Jason Crabtree et al,2495,None,None,Microsoft Corporation
4,IPR2026-00185,Pending,2025-12-30,None,None,16732242,10991097,2021-04-27,2600,Stephen Yip et al,2666,None,None,"Guardant Health, Inc."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19085,DER2017-00004,Pending,2016-12-02,None,None,14726075,12443989,2025-10-14,3600,Jason Broder,3692,None,SIMON TECHNOLOGIES LLC,"Gunther, Christian"
19086,DER2016-00022,Institution Denied,2016-08-25,2016-12-22,None,14580240,9853053,2017-12-26,2800,James John Lupino et al,2814,"3B Technologies, Inc.","3B Technologies, Inc.","Shukh, Alexander"
19087,DER2016-00021,Institution Denied,2016-08-22,2017-10-02,None,14625512,9979460,2018-05-22,2600,Michiel Petrus Lotter et al,2649,Nextivity Inc.,Nextivity Inc.,"Ethertronics, Inc."
19088,DER2016-00003,Institution Denied,2016-01-26,2017-11-07,None,14209123,9067525,2015-06-30,3600,Deyan Ninov et al,3612,FCA US LLC,FCA US LLC,"Sindoni, Michael J. et al."


In [42]:
# I want to check if two columns have the same entries. I compare them easily with pandas
same = proceedings_raw3["patent_owner_data.patent_owner_name"].eq(proceedings_raw3["patent_owner_data.real_party_in_interest_name"])
same.mean() # The mean is 0.214 which means entries are same only the same for 0.214 of entries

np.float64(0.225130890052356)

In [43]:
# I want to see if the entries are different because one of the entries is missing.
# Lets take the two columns from the dataset
a = proceedings_raw3["patent_owner_data.patent_owner_name"]
b = proceedings_raw3["patent_owner_data.real_party_in_interest_name"]

# I can create new values or "masks" for missing cases
both_present = a.notna() & b.notna()
both_missing = a.isna() & b.isna()
one_missing = a.isna() ^ b.isna()

# I also check if both entries are same or different when both entries are present
equal_when_present = a[both_present].eq(b[both_present])
diff_when_present = ~equal_when_present

# I want to see the number of all entry categories
print("Total rows:", len(proceedings_raw3))
print("Both missing:", both_missing.sum())
print("Exactly one missing:", one_missing.sum())
print("from owner:", one_missing[a.isna()].sum())
print("from real party interest:", one_missing[b.isna()].sum())
print("Both present:", both_present.sum())

# I also want to see how the both present entries are divided
print("Equal:", equal_when_present.sum())
print("Different:", diff_when_present.sum())

Total rows: 19100
Both missing: 99
Exactly one missing: 10476
from owner: 10372
from real party interest: 104
Both present: 8525
Equal: 4201
Different: 4324


In [44]:
# Lastly, I check if the differences in different but present entries are because of typo
a2 = a.fillna("").astype(str).str.strip().str.lower()
b2 = b.fillna("").astype(str).str.strip().str.lower()

both_present2 = (a2 != "") & (b2 != "")
equal_when_present2 = a2[both_present2].eq(b2[both_present2])

print("\nCase/space-insensitive, among Both present:")
print("  Equal:", equal_when_present2.sum())
print("  Different:", (~equal_when_present2).sum())
# I remove the " " entries where pandas previously viewed as not missing this reduces the equal entries to 4209 while making different entries to 4249, in total 8458. This is 67 rows less than the original count

# I want to inspect the cases where both present entries mismatch
mismatches = proceedings_raw3.loc[both_present & ~a.eq(b), ["patent_owner_data.patent_owner_name", "patent_owner_data.real_party_in_interest_name"]].head(30)
mismatches.sample(20)


Case/space-insensitive, among Both present:
  Equal: 4209
  Different: 4249


,patent_owner_data.patent_owner_name,patent_owner_data.real_party_in_interest_name
4469,"Leem, Sung Hyun null",Scramoge Technology Ltd.
4461,"Lutter, Robert Pierce",MicroPairing Technologies LLC
4392,Shirazee et al,ePropelled Inc.
4374,Zhidov et al,"FLYPSI, INC."
4462,"Bae, Su Ho null",Scramoge Technology Ltd.
4372,ZHIDOV et al,"FLYPSI, INC."
4219,Chang et al,"Flexiworld Technologies, Inc."
4455,"Lutter, Robert Pierce",MicroPairing Technologies LLC
4339,Chang et al,"Flexiworld Technologies, Inc."
4179,Beyer et al,Ocean Semiconductor LLC


In [45]:
# I do the same for the inventor column and the patent owner.
# Lets take the two columns from the dataset
a = proceedings_raw3["patent_owner_data.inventor_name"]
b = proceedings_raw3["patent_owner_data.patent_owner_name"]

# I can create new values or "masks" for missing cases
both_present = a.notna() & b.notna()
both_missing = a.isna() & b.isna()
one_missing = a.isna() ^ b.isna()

# I also check if both entries are same or different when both entries are present
equal_when_present = a[both_present].eq(b[both_present])
diff_when_present = ~equal_when_present

# I want to see the number of all entry categories
print("Total rows:", len(proceedings_raw3))
print("Both missing:", both_missing.sum())
print("Exactly one missing:", one_missing.sum())
print("from inventor:", one_missing[a.isna()].sum())
print("from owner:", one_missing[b.isna()].sum())
print("Both present:", both_present.sum())

# I also want to see how the both present entries are divided
print("Equal:", equal_when_present.sum())
print("Different:", diff_when_present.sum())

Total rows: 19100
Both missing: 483
Exactly one missing: 10439
from inventor: 451
from owner: 9988
Both present: 8178
Equal: 39
Different: 8139


In [46]:
# Lastly, I check if the differences in different but present entries are because of typo
a3= a.fillna("").astype(str).str.strip().str.lower()
b3 = b.fillna("").astype(str).str.strip().str.lower()

both_present3 = (a3 != "") & (b3 != "")
equal_when_present3 = a3[both_present3].eq(b3[both_present3])

print("\nCase/space-insensitive, among Both present:")
print("  Equal:", equal_when_present3.sum())
print("  Different:", (~equal_when_present3).sum())
# I remove the " " entries where pandas previously viewed as not missing this reduces the equal entries to 4209 while making different entries to 4249, in total 8458. This is 67 rows less than the original count

# I want to inspect the cases where both present entries mismatch
mismatches = proceedings_raw3.loc[both_present & ~a.eq(b), ["patent_owner_data.inventor_name", "patent_owner_data.patent_owner_name"]].head(30)
mismatches.sample(20)


Case/space-insensitive, among Both present:
  Equal: 68
  Different: 8042


,patent_owner_data.inventor_name,patent_owner_data.patent_owner_name
4462,Su Ho Bae,"Bae, Su Ho null"
4372,Ivan ZHIDOV et al,ZHIDOV et al
4334,William Ho CHANG et al,CHANG et al
4463,Su Ho Bae,"Bae, Su Ho null"
4471,Ki Min LEE et al,LEE et al
4078,Sean D. Bradley et al,Bradley et al
4456,Robert Pierce Lutter,"Lutter, Robert Pierce"
4374,Ivan Zhidov et al,Zhidov et al
4381,Ivan ZHIDOV et al,ZHIDOV et al
4224,William Ho Chang et al,Chang et al


In [47]:
# I also check the "patent_owner_data.inventor_name" column which has only real names
example = ["patent_owner_data.inventor_name","patent_owner_data.patent_owner_name","patent_owner_data.real_party_in_interest_name"]
proceedings_raw3[example].sample(20)

,patent_owner_data.inventor_name,patent_owner_data.patent_owner_name,patent_owner_data.real_party_in_interest_name
16950,Roman Chistyakov,Zond LLC,Zond LLC
6593,Ernst Bretschneider,None,NXP B.V. et al.
351,Walter Ernest Power II et al,None,Tethrd LLC
55,Kwok Wai Cheung et al,None,"IngenioSpec, LLC"
16275,Kenneth Schofield et al,Magna Electronics Inc.,Magna Electronics Inc.
6278,Toshiyuki Kurita et al,Kurita et al,"Maxell, Ltd."
5895,Mark S. Blumenkranz et al,None,"AMO Development, LLC et al."
4891,Jung-Fu CHENG et al,None,Telefonaktiebolaget LM Ericsson et al.
5743,Alfonso Campo Camacho et al,None,"TOT Power Control, S.L."
5465,Matti Samuli Halme,None,"WSOU Investments, LLC D/B/A Brazos Licensing a..."


In [48]:
# I conclude the column "patent_owner_data.patent_owner_name" is not necessary and I decide to drop it. I explain why
### Out of 17931 rows only 7892 is non-empty.
### 95 entries of "patent_owner_data.real_party_in_interest_name" is empty while "patent_owner_data.patent_owner_name" is not empty.
### Only 7892 of these are non-empty at the same time as "patent_owner_data.real_party_in_interest_name".
### 3769 of these entries are the same. These are the entries where the firm owns the patent.
### 4056 of these entries are different. "proceedings_raw3[example].sample(20)" command shows us these entries correspond to the inventor names.
### In conclusion information about owners and firms are already preserved in the dataset in different columns in a better written way.
### "patent_owner_data.patent_owner_name" only tells us who owns the patent.
### Furthermore there are lot of typo issues in this column. Some entries consist of all capital letters, some entries are missing "et al." which causes entries to be counted as different. Row 12154. row 14842 are such examples
### One unique information is for example row 12643 where the owner name entry is different from both inventor (empty) and real party in interest.

In [49]:
proceedings_raw3 = proceedings_raw3.drop(columns=["patent_owner_data.patent_owner_name"])

In [50]:
# I also check if "patent_owner_data.application_number_text" and "patent_owner_data.patent_number" consist of unique pairs.
# I find 0 pairs equal and 17930 pairs unique. So I don't need both columns since the patent applications in this data set are already accepted.
c = proceedings_raw3["patent_owner_data.application_number_text"]
d = proceedings_raw3["patent_owner_data.patent_number"]

x = c.notna() & d.notna()

equal = c[x].eq(d[x])
diff = ~equal

print("Equal:", equal.sum())
print("Different:", diff.sum())

Equal: 0
Different: 19092


In [51]:
proceedings_raw3 = proceedings_raw3.drop(columns=["patent_owner_data.application_number_text"])

In [52]:
# I rearrannge the order of the columns for my liking
cols3 = [
    "patent_owner_data.patent_number",
    "patent_owner_data.grant_date",
    "patent_owner_data.technology_center_number",
    "patent_owner_data.group_art_unit_number",
    "patent_owner_data.inventor_name",
    "trial_number",
    "trial_meta_data.trial_status_category",
    "trial_meta_data.petition_filing_date",
    "trial_meta_data.institution_decision_date",
    "trial_meta_data.termination_date",
    "patent_owner_data.real_party_in_interest_name",
    "regular_petitioner_data.real_party_in_interest_name",
]

proceedings_raw4 = proceedings_raw3[cols3]

# I also rename the columns for easier reading
proceedings_raw4.columns = ["patent_number", "grant_date", "technology_center_number", "group_art_unit_number", "inventor_name", "trial_number",
"trial_status_category", "petition_filing_date", "institution_decision_date", "termination_date", "respondent", "petitioner" ]

In [53]:
# I change data types before any save
proceedings_raw4["patent_number"] = proceedings_raw4["patent_number"].astype("string")
proceedings_raw4["grant_date"] = pd.to_datetime(proceedings_raw4["grant_date"], errors="coerce")
proceedings_raw4["technology_center_number"] = pd.to_numeric(proceedings_raw4["technology_center_number"], errors="coerce").astype("Int64")
proceedings_raw4["group_art_unit_number"] = pd.to_numeric(proceedings_raw4["group_art_unit_number"], errors="coerce").astype("Int64")
proceedings_raw4["petition_filing_date"] = pd.to_datetime(proceedings_raw4["petition_filing_date"], errors="coerce")
proceedings_raw4["institution_decision_date"] = pd.to_datetime(proceedings_raw4["institution_decision_date"], errors="coerce")
proceedings_raw4["termination_date"] = pd.to_datetime(proceedings_raw4["termination_date"], errors="coerce")
proceedings_raw4["respondent"] = proceedings_raw4["respondent"].astype("string")
proceedings_raw4["petitioner"] = proceedings_raw4["petitioner"].astype("string")
proceedings_raw4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19100 entries, 0 to 19099
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   patent_number              19093 non-null  string        
 1   grant_date                 19092 non-null  datetime64[ns]
 2   technology_center_number   19079 non-null  Int64         
 3   group_art_unit_number      17859 non-null  Int64         
 4   inventor_name              18166 non-null  object        
 5   trial_number               19100 non-null  object        
 6   trial_status_category      19097 non-null  object        
 7   petition_filing_date       19098 non-null  datetime64[ns]
 8   institution_decision_date  16514 non-null  datetime64[ns]
 9   termination_date           15610 non-null  datetime64[ns]
 10  respondent                 18897 non-null  string        
 11  petitioner                 19038 non-null  string        
dtypes: I

In [54]:
proceedings_raw4 = proceedings_raw4.rename(columns={"technology_center_number":"examiner_tech_center_number","group_art_unit_number":"examiner_art_unit"})

In [55]:
# I select the proceedings until 2026.
start = pd.Timestamp("2012-09-16")
end = pd.Timestamp("2025-12-31")

m_filing = proceedings_raw4["petition_filing_date"].between(start, end, inclusive="both")

proceedings_raw5 = proceedings_raw4[m_filing]

In [56]:
# Extract the leading letters (e.g., IPR, DER, CBM, PGR)
proceedings_raw5["trial_type"] = (
    proceedings_raw5["trial_number"]
      .astype(str)
      .str.extract(r"^([A-Z]+)", expand=False)
)

# Count each type
counts = proceedings_raw5["trial_type"].value_counts(dropna=False)
print(counts)

trial_type
IPR    17928
CBM      602
PGR      539
DER       28
Name: count, dtype: int64


/var/folders/gn/x5brd2hs44x9lc3g72_l7b680000gn/T/ipykernel_1204/1715596727.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  proceedings_raw5["trial_type"] = (


In [58]:
# I want to filter the IPR cases
proceedings_raw5 = proceedings_raw5[proceedings_raw5["trial_type"] == "IPR"]
proceedings_raw5.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17928 entries, 1 to 17928
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   patent_number                17928 non-null  string        
 1   grant_date                   17927 non-null  datetime64[ns]
 2   examiner_tech_center_number  17907 non-null  Int64         
 3   examiner_art_unit            16817 non-null  Int64         
 4   inventor_name                17005 non-null  object        
 5   trial_number                 17928 non-null  object        
 6   trial_status_category        17927 non-null  object        
 7   petition_filing_date         17928 non-null  datetime64[ns]
 8   institution_decision_date    15461 non-null  datetime64[ns]
 9   termination_date             14720 non-null  datetime64[ns]
 10  respondent                   17744 non-null  string        
 11  petitioner                   17872 non-null  s

In [59]:
# I check the patents that have more than one proceeding
pat_trials = proceedings_raw5.groupby("patent_number")["trial_number"].nunique()

share_multi = (pat_trials > 1).mean()
print("Share of patents in multiple trials:", share_multi)

print("Top patents by number of trials:")
print(pat_trials.sort_values(ascending=False).head())


Share of patents in multiple trials: 0.28656665530173886
Top patents by number of trials:
patent_number
7237634    27
6853142    23
6805779    21
7147759    19
RE40264    18
Name: trial_number, dtype: int64


In [60]:
proceedings_raw5["trial_status_category"].value_counts(dropna=False)

trial_status_category
Institution Denied                   5497
Final Written Decision               5337
Terminated-Settled                   4154
Terminated                            873
Final Written Decision - Appealed     577
Discretionary Denial                  463
Trial Instituted                      438
Pending                               365
Terminated-Adverse Judgment            91
Pending Director Review                71
Terminated-Other                       45
Terminated-Dismissed                   16
None                                    1
Name: count, dtype: int64

In [61]:
proceedings_raw5.sample(10)

,patent_number,grant_date,examiner_tech_center_number,examiner_art_unit,inventor_name,trial_number,trial_status_category,petition_filing_date,institution_decision_date,termination_date,respondent,petitioner,trial_type
9734,9124950,2015-09-01,2400,2484,Max Abecassis,IPR2018-01497,Final Written Decision - Appealed,2018-08-01,2019-03-07,2020-03-05,"CustomPlay, LLC","Amazon.com, Inc.",IPR
8685,6397186,2002-05-28,2600,2641,None,IPR2019-00875,Terminated,2019-04-03,NaT,2019-08-20,"SpeakWare, Inc.",Apple Inc.,IPR
5823,8165024,2012-04-24,2400,2471,Andrew Dolganow et al,IPR2021-00888,Terminated-Settled,2021-05-14,NaT,2021-08-10,"Proven Networks, LLC",Microsoft Corp.,IPR
14147,6160359,2000-12-12,2800,<NA>,MARC FLEISCHMANN,IPR2016-00190,Terminated,2015-11-20,2016-03-29,2016-03-29,"Intuitive Building Controls, Inc. (a wholly-ow...",AMX LLC et al.,IPR
6087,8988796,2015-03-24,2800,2872,WEI-YU CHEN,IPR2021-00641,Terminated-Settled,2021-03-09,NaT,2021-05-12,"Largan Precision Co., Ltd.",HP Inc.,IPR
16469,7404660,2008-07-29,2800,2875,Jeffery R. Parker,IPR2014-01094,Institution Denied,2014-07-01,2015-01-13,2015-01-13,Innovative Display Technologies LLC,"LG Display Co., Ltd. et al.",IPR
5969,10045184,2018-08-07,2600,2648,John PADGETT et al,IPR2021-00782,Institution Denied,2021-04-07,2021-10-12,2021-10-12,Carnival Corporation,DeCurtis LLC,IPR
6552,6947810,2005-09-20,2100,2125,Paul W. Skinner,IPR2021-00077,Institution Denied,2020-10-19,2021-03-24,NaT,Vineyard Investigations,E. & J. Gallo Winery,IPR
11106,9538693,2017-01-03,2800,2835,Arthur Kurz et al,IPR2017-02038,Institution Denied,2017-08-31,2018-03-14,NaT,"AK Stamping Company, Inc.","Laird Technologies, Inc.",IPR
7771,9647070,2017-05-09,2800,2892,G.R. Mohan Rao,IPR2020-00292,Terminated-Settled,2019-12-24,NaT,2020-06-29,Greenthread LLC,"Samsung Electronics Co., Ltd.",IPR


In [62]:
proceedings_raw5.to_csv("/Users/cantutuncu/Documents/Bocconi University/THESIS/Data/merged_data/proceedings_final.csv", index = False)